<a href="https://colab.research.google.com/github/Timang419/deep-learning-for-mortgage-/blob/main/script3_last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install hdf5plugin optuna seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 47.0 MB/s eta 0:00:00


In [ ]:
import os
n_cpus = os.cpu_count()
print(f"Available CPUs: {n_cpus}")  # confirm what Colab gave you

# Must be set BEFORE any h5py import
os.environ["HDF5_PLUGIN_MAX_THREADS"] = str(n_cpus)

Available CPUs: 12


In [ ]:
import os, gc, json, warnings, time
# Unlock all allocated CPUs for LZ4 decompression (must be set before h5py import)
os.environ.setdefault("HDF5_PLUGIN_MAX_THREADS",
                      os.environ.get("SLURM_CPUS_PER_TASK", str(os.cpu_count())))
import numpy as np
import pandas as pd
import hdf5plugin                      # MUST import before h5py — registers LZ4 codec
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import joblib
import optuna
import ctypes
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score, log_loss,
    precision_score, recall_score,
)
from sklearn.metrics import roc_curve, auc as sklearn_auc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import shutil as _shutil
import threading
from sklearn.metrics import precision_recall_fscore_support
import shutil
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore', category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

set seed

In [ ]:
SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
  GPU  : NVIDIA A100-SXM4-80GB
  VRAM : 85.1 GB


In [ ]:
if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability()
    AMP_DTYPE = torch.bfloat16 if _cap[0] >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float16

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


sample

In [ ]:
# SSO        = 'linc6017'
# #BASE       = f'/data/math-deep-learning-course/{SSO}'  # ARC only

# #HDF5_DIR   = '/mnt/local-scratch/merged'                                        # NVMe — fast reads
# OUTPUT_DIR = '/content/drive/MyDrive/dissertation/models/sample_test'           # Drive — survives disconnect
# #SCRATCH    = '/mnt/local-scratch'
# HDF5_DIR   = '/content/merged'
# SCRATCH    = '/content'                                            # local NVMe for Optuna DB
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# DRIVE_HDF5 = '/content/drive/MyDrive/dissertation/hdf5_sample_fullmerged'       # source on Drive

# # ── Copy sample data to NVMe at startup ──────────────────────────────────────
# import shutil
# os.makedirs(HDF5_DIR, exist_ok=True)
# for fname in os.listdir(DRIVE_HDF5):
#     dst = os.path.join(HDF5_DIR, fname)
#     if not os.path.exists(dst):
#         print(f'  copying {fname}...')
#         shutil.copy2(os.path.join(DRIVE_HDF5, fname), dst)
# print('Data ready on NVMe.')

# N_MERGED     = 10
# MERGED_PATHS = [os.path.join(HDF5_DIR, f'train_shard_{i}_merged.h5') for i in range(N_MERGED)]
# HPO_SHARD    = os.path.join(HDF5_DIR, 'train_shard_0.h5')
# VAL_H5       = os.path.join(HDF5_DIR, 'val.h5')
# TEST_H5      = os.path.join(HDF5_DIR, 'test.h5')
# COLS_JSON    = os.path.join(HDF5_DIR, 'feature_cols.json')
# OPTUNA_DB    = f'sqlite:///{SCRATCH}/optuna_study_full.db'

full

In [ ]:
SSO        = 'linc6017'

HDF5_DIR   = '/mnt/local-scratch/merged'
OUTPUT_DIR = '/content/drive/MyDrive/dissertation/models/full_run'
SCRATCH    = '/mnt/local-scratch'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HDF5_DIR, exist_ok=True)

DRIVE_HDF5 = '/content/drive/MyDrive/dissertation/hdf5_full_merged'

# ── Step 1: copy all 10 merged shards to NVMe (290GB, ~30 min) ───────────────
def _copy(fname):
    dst = os.path.join(HDF5_DIR, fname)
    if not os.path.exists(dst):
        print(f'  copying {fname}...', flush=True)
        t0 = time.time()
        shutil.copy2(os.path.join(DRIVE_HDF5, fname), dst)
        elapsed = max(time.time() - t0, 0.001)
        gb = os.path.getsize(dst) / 1e9
        print(f'  {fname} done in {elapsed:.0f}s ({gb:.1f}GB @ {gb*1000/elapsed:.0f}MB/s)', flush=True)

merged_fnames = [f'train_shard_{i}_merged.h5' for i in range(10)]
print('Copying merged shards to NVMe...')
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(_copy, merged_fnames))
print('All merged shards on NVMe.')

# ── Step 2: extract HPO subsets from Drive (fast — first N rows only) ─────────
def extract_hpo_subset(drive_path, local_path, max_rows):
    if os.path.exists(local_path):
        return
    fname = os.path.basename(local_path)
    print(f'  extracting {fname} ({max_rows/1e6:.0f}M rows)...', flush=True)
    t0 = time.time()
    with h5py.File(drive_path, 'r') as src:
        n = min(max_rows, src['X'].shape[0])
        with h5py.File(local_path, 'w') as dst:
            for key in src.keys():
                data = src[key][:n]
                chunks = (min(10000, n), data.shape[1]) if data.ndim == 2 \
                         else (min(10000, n),)
                dst.create_dataset(key, data=data,
                                   chunks=chunks, **hdf5plugin.LZ4())
    print(f'  {fname} done in {time.time()-t0:.0f}s', flush=True)

# HPO shard: only first 60M rows needed
HPO_LOCAL = os.path.join(HDF5_DIR, 'train_shard_0_hpo.h5')
extract_hpo_subset(
    os.path.join(DRIVE_HDF5, 'train_shard_0.h5'),
    HPO_LOCAL, max_rows=60_000_000)

# Val: only first 25M rows needed for HPO
VAL_LOCAL = os.path.join(HDF5_DIR, 'val_hpo.h5')
extract_hpo_subset(
    os.path.join(DRIVE_HDF5, 'val.h5'),
    VAL_LOCAL, max_rows=25_000_000)

# Non-HDF5 files
for fname in ['feature_cols.json', 'scaler.pkl']:
    _copy(fname)


print('NVMe ready.')

print(f'  Merged shards : 290 GB on NVMe')
print(f'  HPO shard     : ~8 GB on NVMe (60M rows)')
print(f'  Val HPO       : ~4 GB on NVMe (25M rows)')
print(f'  test.h5       : ~108 GB on /content (fast reads)')

# ── Paths ─────────────────────────────────────────────────────────────────────
N_MERGED     = 10
MERGED_PATHS = [os.path.join(HDF5_DIR, f'train_shard_{i}_merged.h5')
                for i in range(N_MERGED)]
HPO_SHARD    = HPO_LOCAL                                    # 60M row subset
VAL_H5       = VAL_LOCAL                                    # 25M row subset
TEST_H5      = os.path.join(DRIVE_HDF5, 'test.h5')         # Drive — read once
COLS_JSON    = os.path.join(HDF5_DIR, 'feature_cols.json')
OPTUNA_DB    = f'sqlite:///{SCRATCH}/optuna_study_full.db'

Copying merged shards to NVMe...
  copying train_shard_0_merged.h5...
  copying train_shard_1_merged.h5...
  copying train_shard_2_merged.h5...
  copying train_shard_3_merged.h5...
  train_shard_3_merged.h5 done in 615s (30.7GB @ 50MB/s)
  copying train_shard_4_merged.h5...
  train_shard_0_merged.h5 done in 619s (30.7GB @ 50MB/s)
  copying train_shard_5_merged.h5...
  train_shard_2_merged.h5 done in 641s (30.8GB @ 48MB/s)
  copying train_shard_6_merged.h5...
  train_shard_1_merged.h5 done in 662s (30.6GB @ 46MB/s)
  copying train_shard_7_merged.h5...
  train_shard_6_merged.h5 done in 606s (30.8GB @ 51MB/s)
  copying train_shard_8_merged.h5...
  train_shard_4_merged.h5 done in 641s (30.5GB @ 48MB/s)
  copying train_shard_9_merged.h5...
  train_shard_5_merged.h5 done in 650s (30.7GB @ 47MB/s)
  train_shard_7_merged.h5 done in 677s (30.7GB @ 45MB/s)
  train_shard_9_merged.h5 done in 611s (30.7GB @ 50MB/s)
  train_shard_8_merged.h5 done in 631s (30.6GB @ 48MB/s)
All merged shards on NVMe.


In [ ]:
N_LOAD_WORKERS = 4

NUM_CLASSES = 7
STATE_NAMES = ['Current', 'D30', 'D60', 'D90+', 'Foreclosure', 'REO', 'PaidOff']

REPORTING_PERIOD_COL = 'Reporting_Period_Int'   # bookkeeping only — excluded from model

# HPO grids
FIXED_LR = 1e-3
ZERO_LAYER_WD_GRID = [1e-5, 1e-4, 1e-3, 1e-2]
# for the data to use for hpo
HPO_ROWS = 60_000_000
# chunk size that use to push to vram
CHUNK_ROWS = 60_000_000
HPO_VAL_ROWS = 25_000_000

In [ ]:
def get_model_feature_indices(feature_cols):
    """Drop Reporting_Period_Int — it's a bookkeeping column, not a model input."""
    return [i for i, c in enumerate(feature_cols) if c != REPORTING_PERIOD_COL]

def recover_current_state_from_cols(h5_path, feature_cols):
    state_cols = sorted(
        [c for c in feature_cols
         if c.startswith('state_') and c.split('_')[-1].isdigit()],
        key=lambda c: int(c.split('_')[-1])
    )
    if len(state_cols) < 2:
        print('[WARN] recover_current_state: no state_* columns found.')
        return None
    idx = [feature_cols.index(c) for c in state_cols]

    # Memory-safe chunked read
    curr_states = []
    with h5py.File(h5_path, 'r') as f:
        n_rows = f['X'].shape[0]
        for start in range(0, n_rows, CHUNK_ROWS):
            end = min(start + CHUNK_ROWS, n_rows)
            X_raw = f['X'][start:end]
            state_ohe = X_raw[:, idx]
            del X_raw
            curr_states.append(np.argmax(state_ohe, axis=1).astype(np.int32))

    return np.concatenate(curr_states)

In [ ]:
def _load_one(h5_path, feat_indices, max_rows=None):
    """Read HDF5 → (X float32, y int64) using parallel file handles.

    Each thread opens its own h5py.File and reads a contiguous row slice,
    bypassing HDF5 single-threaded chunk dispatch (~25ms/chunk overhead).
    Benchmarked 2.5x speedup on 8 threads vs serial on same hardware.
    max_rows: if set, only load first max_rows rows (used for HPO subsampling).
    """
    n_threads = int(os.environ.get('HDF5_PLUGIN_MAX_THREADS', '4'))

    t0 = time.time()
    with h5py.File(h5_path, 'r') as f:
        total = f['X'].shape[0]
    n_rows  = total if max_rows is None else min(max_rows, total)
    n_feats = len(feat_indices)

    X = np.empty((n_rows, n_feats), dtype=np.float32)
    y = np.empty(n_rows,            dtype=np.int64)

    bounds = np.linspace(0, n_rows, n_threads + 1, dtype=int)
    ranges = [(int(bounds[i]), int(bounds[i + 1]))
              for i in range(n_threads) if bounds[i] < bounds[i + 1]]

    def _read_slice(r_start, r_end):
        with h5py.File(h5_path, 'r') as f:
            X_raw          = f['X'][r_start:r_end]
            X[r_start:r_end] = X_raw[:, feat_indices]
            del X_raw
            y[r_start:r_end] = f['y'][r_start:r_end].astype(np.int64)

    t_read0 = time.time()
    with ThreadPoolExecutor(max_workers=n_threads) as pool:
        list(pool.map(lambda r: _read_slice(*r), ranges))
    elapsed = time.time() - t_read0

    print(f'      _load_one: {os.path.basename(h5_path)}  '
          f'{n_threads}t  {elapsed:.1f}s  '
          f'({X.nbytes / 1e6 / elapsed:.0f} MB/s)', flush=True)
    return X, y

def _to_tensor_ds(X, y):
    return TensorDataset(torch.from_numpy(X), torch.from_numpy(y))

def load_h5_parallel(paths_and_indices, n_workers=N_LOAD_WORKERS, max_rows=None):
    """Load multiple HDF5 files simultaneously with a thread pool."""
    n       = len(paths_and_indices)
    results = [None] * n
    errors  = [None] * n

    def _worker(i, path, idx):
        try:
            X, y       = _load_one(path, idx, max_rows)
            results[i] = _to_tensor_ds(X, y)
            print(f'  [t{i:02d}] {os.path.basename(path):35s} '
                  f'{X.shape[0]:>10,} rows  {X.nbytes/1e6:6.0f} MB', flush=True)
        except Exception as e:
            errors[i] = e

    with ThreadPoolExecutor(max_workers=min(n_workers, n)) as pool:
        futures = [pool.submit(_worker, i, p, idx)
                   for i, (p, idx) in enumerate(paths_and_indices)]
        for fut in futures:
            fut.result()

    for i, e in enumerate(errors):
        if e is not None:
            raise RuntimeError(f'load_h5_parallel failed at index {i}') from e
    gc.collect()
    return results

In [ ]:
class ChunkPrefetcher:
    """
    Prefetches one (shard_path, start, end) chunk from HDF5 in a background
    thread while the GPU trains on the current chunk.

    Timeline:
      Thread : [load chunk i][load chunk i+1][load chunk i+2]...
      GPU    :        [train chunk i  ][train chunk i+1  ]...
                      ^ get() returns here (already done)
    """
    def __init__(self, feat_indices):
        self._feat_indices = feat_indices
        self._result = None
        self._thread = None
        self._exc    = None

    def start(self, path, start, end):
        self._result = self._exc = None
        feat_idx  = self._feat_indices
        n_threads = int(os.environ.get('HDF5_PLUGIN_MAX_THREADS', '4'))
        n_rows    = end - start
        X = np.empty((n_rows, len(feat_idx)), dtype=np.float32)
        y = np.empty(n_rows,                  dtype=np.int64)

        bounds = np.linspace(0, n_rows, n_threads + 1, dtype=int)
        ranges = [(int(bounds[i]), int(bounds[i + 1]))
                  for i in range(n_threads) if bounds[i] < bounds[i + 1]]

        def _work():
            try:
                def _read_slice(r_start, r_end):
                    abs_s = start + r_start
                    abs_e = start + r_end
                    with h5py.File(path, 'r') as f:
                        X_raw = f['X'][abs_s:abs_e]
                        X[r_start:r_end] = X_raw[:, feat_idx]
                        del X_raw
                        y[r_start:r_end] = f['y'][abs_s:abs_e].astype(np.int64)
                with ThreadPoolExecutor(max_workers=n_threads) as pool:
                    list(pool.map(lambda r: _read_slice(*r), ranges))
                self._result = (X, y)
            except Exception as e:
                self._exc = e

        self._thread = threading.Thread(target=_work, daemon=True)
        self._thread.start()

    def get(self):
        if self._thread is not None:
            self._thread.join()
        if self._exc is not None:
            raise self._exc
        return self._result   # (X_np float32, y_np int64)



PERFORMANCE METRICS

In [ ]:
def compute_metrics(y_true, y_pred, y_proba, name=''):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Parallel chunked log_loss
    CHUNK = 10_000_000
    n = len(y_true)
    chunks = [(start, min(start + CHUNK, n))
              for start in range(0, n, CHUNK)]

    def _chunk_ll(args):
        start, end = args
        return log_loss(y_true[start:end], y_proba[start:end],
                       labels=list(range(NUM_CLASSES))) * (end - start)

    with ThreadPoolExecutor(max_workers=12) as pool:
        ll_parts = list(pool.map(_chunk_ll, chunks))
    ll = sum(ll_parts) / n

    if name:
        print(f'\n{name} — Scalar Metrics:')
        print(f'  Neg. avg log-likelihood (↓) : {ll:.5f} ')
        print(f'  Macro-F1                    : {f1:.5f} ')
        print(f'  Accuracy                    : {acc:.5f} ')
    return dict(log_loss=ll, macro_f1=f1, accuracy=acc)


def compute_per_class_metrics(y_true, y_pred, y_proba):
    # One single pass over the data for all classes and metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)),
        average=None, zero_division=0)

    return pd.DataFrame({
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'support':   support.astype(int)
    }, index=STATE_NAMES)


def compute_auc_matrix(y_true, y_proba, current_states):
    import ctypes
    M = np.full((NUM_CLASSES, NUM_CLASSES), np.nan)

    for u in range(NUM_CLASSES):
        mask = (current_states == u)
        n_u = mask.sum()
        if n_u < 10:
            continue

        y_u     = y_true[mask]
        proba_u = y_proba[mask]

        MAX_SAMPLES = 10_000_000
        if len(y_u) > MAX_SAMPLES:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(y_u), MAX_SAMPLES, replace=False)
            y_u     = y_u[idx]
            proba_u = proba_u[idx]

        for v in range(NUM_CLASSES):
            binary = (y_u == v).astype(np.int8)
            n_pos = int(binary.sum())
            n_neg = int(len(y_u)) - n_pos

            if n_pos == 0 or n_neg == 0:
                M[u, v] = np.nan
                del binary
                continue

            try:
                M[u, v] = roc_auc_score(binary, proba_u[:, v])
            except Exception:
                M[u, v] = np.nan
            finally:
                del binary

        del y_u, proba_u, mask
        gc.collect()
        ctypes.CDLL("libc.so.6").malloc_trim(0)

    return M


def build_transition_matrix(current_states, next_states, n=NUM_CLASSES):
    """Row-normalised empirical P(s'|s)."""
    T = np.zeros((n, n), dtype=np.float64)

    # Vectorized counting (Replaces the slow for-loop)
    mask = (current_states >= 0) & (current_states < n) & (next_states >= 0) & (next_states < n)
    np.add.at(T, (current_states[mask], next_states[mask]), 1)

    row_sums = T.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return T / row_sums

# **plot**

In [ ]:
def plot_transition_matrix(T, title, save_path):
    df = pd.DataFrame(T, index=STATE_NAMES, columns=STATE_NAMES)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(df, annot=True, fmt='.3f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Next State'); ax.set_ylabel('Current State')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f'  Saved: {save_path}')


def plot_auc_matrix(M, title, save_path):
    """7×7 AUC matrix — main discrimination result."""
    df    = pd.DataFrame(M, index=STATE_NAMES, columns=STATE_NAMES)
    annot = df.map(lambda x: f'{x:.2f}' if not np.isnan(x) else '—')
    mask  = np.isnan(M)

    # Remove absorbing states — never appear as current state
    rows_to_keep = [s for s in STATE_NAMES if s not in ['REO', 'PaidOff']]
    row_idx      = [STATE_NAMES.index(s) for s in rows_to_keep]
    df    = df.loc[rows_to_keep]
    annot = annot.loc[rows_to_keep]
    mask  = mask[row_idx, :]

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(df, annot=annot, fmt='', cmap='RdPu',
                linewidths=0.5, ax=ax, vmin=0.50, vmax=1.0,
                mask=mask, cbar_kws={'label': 'AUC[u,v]'})
    ax.set_facecolor('#e0e0e0')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Next State (v)'); ax.set_ylabel('Current State (u)')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f'  Saved: {save_path}')


def plot_training_curves(history, save_path):
    epochs_r = range(1, len(history['tr_loss']) + 1)
    has_val  = 'val_loss' in history and len(history.get('val_loss', [])) > 0
    n_panels = 3 if has_val else 1
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4))
    if n_panels == 1:
        axes = [axes]
    axes[0].plot(epochs_r, history['tr_loss'], label='Train', color='steelblue')
    if has_val:
        axes[0].plot(epochs_r, history['val_loss'], label='Val')
    axes[0].set_title('Cross-Entropy Loss ')
    axes[0].legend()
    if has_val:
        axes[1].plot(epochs_r, history['val_f1'])
        axes[1].set_title('Val Macro-F1')
        axes[2].plot(epochs_r, history['val_acc'])
        axes[2].set_title('Val Accuracy')
    for ax in axes:
        ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f'  Saved: {save_path}')



def plot_comparison_bar(results_df, save_path):
    metrics = ['log_loss', 'macro_f1', 'accuracy']
    labels  = ['Neg. Log-Likelihood',
               'Macro-F1',
               'Accuracy']
    colors  = ['#C44E52', '#4C72B0', '#55A868']
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, metric, label, color in zip(axes, metrics, labels, colors):
        vals = results_df[metric]
        bars = ax.bar(results_df['Model'], vals, color=color, alpha=0.85)
        ax.set_title(label, fontsize=10)
        ax.tick_params(axis='x', rotation=20)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.003,
                    f'{bar.get_height():.4f}',
                    ha='center', va='bottom', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    plt.suptitle('Test Set Performance Comparison (2021–2025)', fontsize=13, y=1.01)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(); print(f'  Saved: {save_path}')


def plot_auc_matrices_side_by_side(auc_dict, save_path):
    n = len(auc_dict)
    fig, axes = plt.subplots(1, n, figsize=(9 * n + 1, 7))
    if n == 1: axes = [axes]
    for ax, (title, M) in zip(axes, auc_dict.items()):
        df    = pd.DataFrame(M, index=STATE_NAMES, columns=STATE_NAMES)
        annot = df.map(lambda x: f'{x:.2f}' if not np.isnan(x) else '—')
        mask  = np.isnan(M)

        # Remove absorbing states — never appear as current state
        rows_to_keep = [s for s in STATE_NAMES if s not in ['REO', 'PaidOff']]
        row_idx      = [STATE_NAMES.index(s) for s in rows_to_keep]
        df    = df.loc[rows_to_keep]
        annot = annot.loc[rows_to_keep]
        mask  = mask[row_idx, :]

        sns.heatmap(df, annot=annot, fmt='', cmap='RdPu',
                    linewidths=0.4, ax=ax, vmin=0.5, vmax=1.0,
                    mask=mask, cbar=(ax == axes[-1]),
                    cbar_kws={'label': 'AUC[u,v]'})
        ax.set_facecolor('#e0e0e0')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Next State (v)')
    axes[0].set_ylabel('Current State (u)')
    plt.suptitle('AUC Matrix — Test Set (2021–2025)\n'
                 'AUC[u,v] = binary AUC for transition u→v', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {save_path}')


def plot_transition_matrices_side_by_side(matrices_dict, save_path):
    n = len(matrices_dict)
    fig, axes = plt.subplots(1, n, figsize=(9 * n + 1, 8))
    if n == 1: axes = [axes]
    for ax, (title, T) in zip(axes, matrices_dict.items()):
        df = pd.DataFrame(T, index=STATE_NAMES, columns=STATE_NAMES)
        sns.heatmap(df, annot=True, fmt='.3f', cmap='YlOrRd',
                    linewidths=0.4, ax=ax, cbar=False, vmin=0, vmax=1)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Next State')
    axes[0].set_ylabel('Current State')
    plt.suptitle('Predicted Transition Matrices — Test Set (2021–2025)', fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {save_path}')

def plot_roc_curves(probas_dict, y_true, current_states, save_path):
    """
    ROC curves for key transitions, all models overlaid.
    Replicates Justin paper Figure 17 style.
    probas_dict: {'Model name': y_proba array}
    y_true: true labels array
    """

    # Key transitions to plot — economically meaningful
    transitions = [
        (0, 6, 'Current → PaidOff'),
        (0, 1, 'Current → D30'),
        (1, 0, 'D30 → Current'),
        (3, 4, 'D90+ → Foreclosure'),
    ]

    fig, axes = plt.subplots(1, len(transitions), figsize=(20, 5))
    colors = ['red', 'blue', 'green']

    for ax, (from_state, to_state, title) in zip(axes, transitions):
        mask = (current_states == from_state)
        if mask.sum() < 10:
            ax.set_title(f'{title}\n(insufficient data)')
            continue
        y_bin = (y_true[mask] == to_state).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == mask.sum():
            ax.set_title(f'{title}\n(no variance)')
            continue
        for (model_name, y_proba), color in zip(probas_dict.items(), colors):
            proba_transition = y_proba[mask, to_state]
            fpr, tpr, _ = roc_curve(y_bin, proba_transition)
            roc_auc = sklearn_auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color,
                    label=f'{model_name} (AUC={roc_auc:.3f})')
        ax.plot([0,1], [0,1], 'k--', alpha=0.5)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.suptitle('ROC Curves — Test Set (2021–2025)', fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {save_path}')



# **model1:Baseline**

In [ ]:
class ConstantBaseline:
    def __init__(self):
        self.T = None

    def fit(self, shard_paths, feature_cols):
        print('  Building empirical T from training shards ...')
        BASELINE_CHUNK = 40_000_000
        state_idx = [feature_cols.index(c)
                     for c in sorted(
                         [c for c in feature_cols
                          if c.startswith('state_') and c.split('_')[-1].isdigit()],
                         key=lambda c: int(c.split('_')[-1]))]

        def _process_shard(path):
            local_counts = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
            with h5py.File(path, 'r') as f:
                n_rows = f['X'].shape[0]
                for start in range(0, n_rows, BASELINE_CHUNK):
                    end = min(start + BASELINE_CHUNK, n_rows)
                    y_s       = f['y'][start:end].astype(np.int64)
                    X_raw     = f['X'][start:end]
                    state_ohe = X_raw[:, state_idx]
                    del X_raw; gc.collect()
                    curr = np.argmax(state_ohe, axis=1).astype(np.int32)
                    mask = ((curr >= 0) & (curr < NUM_CLASSES) &
                            (y_s  >= 0) & (y_s  < NUM_CLASSES))
                    np.add.at(local_counts, (curr[mask], y_s[mask]), 1)
            print(f'    processed {os.path.basename(path)}', flush=True)
            return local_counts

        n_workers = 4
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            all_counts = list(pool.map(_process_shard, shard_paths))

        counts = np.sum(all_counts, axis=0)
        row_sums = counts.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        self.T = counts / row_sums
        print(f'  T built. Total observations: {counts.sum():,}')
        print(pd.DataFrame(self.T, index=STATE_NAMES,
                           columns=STATE_NAMES).round(4).to_string())

    def predict_proba(self, current_states):
        return self.T[current_states]

    def predict(self, current_states):
        return np.argmax(self.T[current_states], axis=1)

    def evaluate_on_h5(self, h5_path, feature_cols):
        state_idx = [feature_cols.index(c)
                     for c in sorted(
                         [c for c in feature_cols
                          if c.startswith('state_') and c.split('_')[-1].isdigit()],
                         key=lambda c: int(c.split('_')[-1]))]
        if len(state_idx) < 2:
            print('[WARN] evaluate_on_h5: no state_* columns found.')
            return None, None, None, None

        y_all, curr_st_all = [], []
        with h5py.File(h5_path, 'r') as f:
            n_rows = f['X'].shape[0]
            for start in range(0, n_rows, CHUNK_ROWS):
                end = min(start + CHUNK_ROWS, n_rows)
                y_all.append(f['y'][start:end].astype(np.int64))
                X_raw     = f['X'][start:end]
                state_ohe = X_raw[:, state_idx]
                del X_raw; gc.collect()
                curr_st_all.append(np.argmax(state_ohe, axis=1).astype(np.int32))

        y       = np.concatenate(y_all)
        curr_st = np.concatenate(curr_st_all)
        return y, self.predict(curr_st), self.predict_proba(curr_st), curr_st

# **model 2 and 3: zero hidden layer and deep Neural Network**

In [ ]:
class MortgageNet(nn.Module):
    """
    Feed-forward NN.
    depth=0 → single linear layer (≡ multinomial logistic regression, Model 2)
    depth>0 → MortgageNet (Model 3)
    """
    def __init__(self, input_dim, num_classes=NUM_CLASSES,
                 depth=5, hidden_dim=256, dropout_rate=0.3, norm_type='batch'):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(depth):
            layers.append(nn.Linear(d, hidden_dim))
            if norm_type == 'batch':
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif norm_type == 'layer':
                layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_rate))
            d = hidden_dim
        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer  = nn.Linear(d, num_classes)

    def forward(self, x):
        return self.output_layer(self.hidden_layers(x))


In [ ]:
def make_optimizer(model, lr, weight_decay):
    """
    AdamW with parameter group split (Kosson et al. 2023):
      ndim >= 2 (weight matrices): subject to weight decay
      ndim <  2 (biases, BN γ/β): excluded from weight decay
    """
    decay    = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]
    return torch.optim.AdamW(
        [{'params': decay,    'weight_decay': weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=lr,
    )


class EarlyStopping:
    def __init__(self, patience=5, delta=1e-4, path=None):
        self.patience  = patience
        self.delta     = delta
        self.path      = path or os.path.join(OUTPUT_DIR, 'tmp_checkpoint.pt')
        self.best_loss = np.inf
        self.counter   = 0
        self.stop      = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def load_best(self, model):
        model.load_state_dict(torch.load(self.path, map_location=DEVICE,
                                         weights_only=True))
        return model

In [ ]:
@torch.no_grad()
def evaluate_nn(model, loader, criterion):
    """Evaluate on a DataLoader (e.g. for HPO val)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0 # total loss ,total number fo the model higerd predicted state macthed the actual label, n i stotal number of row process so far
    # store predicted label, true label, seven state transition prob vector
    all_preds, all_labels, all_probs = [], [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                            enabled=DEVICE.type == 'cuda'):
            logits = model(X_b) # feed input to get output vector but this is the 7 state logit that havent normalized to prob
            loss   = criterion(logits, y_b) # apply softmax then compute log loss
        logits   = logits.float() # upcast back to float 32
        preds    = logits.argmax(dim=1)
        probs    = F.softmax(logits, dim=1) # vectro that normalized to prob
        total_loss += loss.item() * len(y_b)
        correct    += (preds == y_b).sum().item()
        n          += len(y_b)
        all_preds.append(preds.cpu()) # move result that computed in gpu back to cpu and appen to all pred.
        all_labels.append(y_b.cpu())
        all_probs.append(probs.cpu())
    all_preds  = torch.cat(all_preds).numpy() # concatenate all batch reuslt into one: from [tensor[],...,tensor[]] to tensor[...]
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.cat(all_probs).numpy()
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, correct / n, f1, all_preds, all_labels, all_probs # however for latest versin this is dead code


@torch.no_grad()
def _eval_gpu_tensors(model, X, y, crit, bs=32768):
    """
    Evaluate directly on GPU tensors without DataLoader.
    Used in train_nn (HPO) where both train and val are already in VRAM.
    Returns (loss, accuracy, macro_f1).
    """
    model.eval()
    total_loss, correct, n = 0.0, 0, X.shape[0]
    all_preds = []
    for i in range(0, n, bs):
        X_b = X[i:i + bs]; y_b = y[i:i + bs]
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                            enabled=DEVICE.type == 'cuda'):#DEVICE.type == 'cuda'
            logits = model(X_b)
            loss   = crit(logits, y_b)
        logits_f = logits.float()
        preds    = logits_f.argmax(dim=1)
        total_loss += loss.item() * len(y_b)
        correct    += (preds == y_b).sum().item()
        all_preds.append(preds.cpu())
    f1 = f1_score(
        y.cpu().numpy(),
        torch.cat(all_preds).numpy(),
        average='macro', zero_division=0)
    return total_loss / n, correct / n, f1

# dead code in latest version since combined train and val
def evaluate_h5_chunked_loss_only(h5_path, feat_indices, model, crit):
    """
    Val evaluation — synchronous chunked reads.
    ThreadPoolExecutor removed: background thread was CPU-starved,
    causing 160s reads vs 22s in main thread (diagnosed via per-chunk timing).
    Returns scalar loss.
    """
    torch.cuda.empty_cache()
    model.eval()
    total_loss, n = 0.0, 0

    with h5py.File(h5_path, 'r') as f:
        n_rows = f['X'].shape[0]

    chunk_starts = list(range(0, n_rows, CHUNK_ROWS))

    def _load_chunk(start, end):
        with h5py.File(h5_path, 'r') as f:
            X_raw = f['X'][start:end]
            X_cpu = X_raw[:, feat_indices]
            del X_raw   # crucial RAM saver
            y_cpu = f['y'][start:end].astype(np.int64)
        return X_cpu, y_cpu

    with torch.no_grad():
        for i, start in enumerate(chunk_starts):
            t0 = time.time()
            X_cpu, y_cpu = _load_chunk(start, min(start + CHUNK_ROWS, n_rows))
            t_io = time.time() - t0

            t0    = time.time()
            X_gpu = torch.from_numpy(X_cpu).to(DEVICE)
            y_gpu = torch.from_numpy(y_cpu).to(DEVICE)
            t_h2d = time.time() - t0
            del X_cpu, y_cpu

            t0 = time.time()
            for j in range(0, len(X_gpu), 32768):
                X_b    = X_gpu[j:j + 32768]
                y_b    = y_gpu[j:j + 32768]
                logits = model(X_b)
                loss   = crit(logits, y_b)
                total_loss += loss.item() * len(y_b)
                n          += len(y_b)
            t_gpu = time.time() - t0

            print(f'    [val chunk {i}] io_wait={t_io:.1f}s  '
                  f'h2d={t_h2d:.2f}s  gpu={t_gpu:.2f}s', flush=True)

            del X_gpu, y_gpu
            torch.cuda.empty_cache()

    return total_loss / n

# for final evaluation on test set
def evaluate_h5_chunked(h5_path, feat_indices, model, crit):
    """
    Full evaluation (loss + accuracy + F1 + preds + probs) in CHUNK_ROWS chunks.
    Used for test-set evaluation at the end.
    F1 computed on full concatenated arrays — correct for macro-F1.
    """
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds_lst, all_labels_lst, all_probs_lst = [], [], []
    with h5py.File(h5_path, 'r') as f:
        n_rows = f['X'].shape[0]
    for start in range(0, n_rows, CHUNK_ROWS):
        end = min(start + CHUNK_ROWS, n_rows)
        with h5py.File(h5_path, 'r') as f:
            X_raw   = f['X'][start:end]
            X_chunk = X_raw[:, feat_indices]
            del X_raw   # crucial RAM saver
            y_chunk = f['y'][start:end].astype(np.int64)
        X_gpu = torch.from_numpy(X_chunk).to(DEVICE)
        y_gpu = torch.from_numpy(y_chunk).to(DEVICE)
        del X_chunk, y_chunk

        with torch.no_grad():
            for j in range(0, len(X_gpu), 32768):
                X_b    = X_gpu[j:j + 32768]
                y_b    = y_gpu[j:j + 32768]
                logits = model(X_b)
                loss   = crit(logits, y_b)
                preds  = logits.argmax(dim=1)
                probs  = F.softmax(logits, dim=1)
                total_loss += loss.item() * len(y_b)
                correct    += (preds == y_b).sum().item()
                n          += len(y_b)
                all_preds_lst.append(preds.cpu())
                all_labels_lst.append(y_b.cpu())
                all_probs_lst.append(probs.cpu())

        del X_b, y_b, logits

        del X_gpu, y_gpu
        gc.collect()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f'    eval chunk {start:>12,}-{end:>12,} / {n_rows:,}', flush=True)

    all_preds  = torch.cat(all_preds_lst).numpy()
    all_labels = torch.cat(all_labels_lst).numpy()
    all_probs  = torch.cat(all_probs_lst).numpy()
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, correct / n, f1, all_preds, all_labels, all_probs

In [ ]:
# used for hpo only
def train_nn(cfg, X_tr, y_tr, X_val, y_val,
             save_path=None, verbose=True, trial=None):
    # cfg is the dictionary that store that hypeparameter configuaration for a specific trial
    save_path = save_path or os.path.join(OUTPUT_DIR, 'tmp_best_model.pt') # directory to saved best model weight
    n_feats   = X_tr.shape[1] # num of feature
    n_tr      = X_tr.shape[0] # num of observation is 60m fixed fo rhpo only
    bs        = cfg['batch_size']

    model = MortgageNet(
        input_dim=n_feats, depth=cfg['depth'], hidden_dim=cfg['hidden_dim'],
        dropout_rate=cfg['dropout'], norm_type=cfg['norm_type'],
    ).to(DEVICE)
    opt  = make_optimizer(model, cfg['lr'], cfg['weight_decay'])
    crit = nn.CrossEntropyLoss()
    es   = EarlyStopping(patience=cfg['patience'], path=save_path) # howver no early stopping in latst version so early stopping with never trigger but this will show the best epoch that achieve the lowest loss
    history = {k: [] for k in ['tr_loss', 'val_loss', 'val_acc', 'val_f1']}
    # for hpo fixed 10 epoch
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        perm           = torch.randperm(n_tr, device=DEVICE) # random shuffle
        total, n_total = 0.0, 0
        for i in range(0, n_tr - bs + 1, bs):      # loop for each epoch [0,16384,32768,..],note this implement the drop last,so actua train is 59991424
            idx  = perm[i:i + bs] # the random shuffle index
            X_b  = X_tr[idx]; y_b = y_tr[idx] # the x and y after radnom shuffle for this batch
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                                enabled=DEVICE.type == 'cuda'):###DEVICE.type == 'cuda'
                logits = model(X_b)
                loss   = crit(logits, y_b)
            loss.backward() # backprogation compute the weghts
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # clip the gradient to avoid gradient explode
            opt.step() # weight update
            total   += loss.item() * bs # total loss calculation
            n_total += bs # count how many onbservation up to now been processed
        tr_loss = total / n_total

        vl, va, vf1 = _eval_gpu_tensors(model, X_val, y_val, crit, bs=32768) # vl->validation loss,va-> accuracy, vf1-> f1 score
        history['tr_loss'].append(tr_loss); history['val_loss'].append(vl)
        history['val_acc'].append(va);      history['val_f1'].append(vf1)

        if verbose: # if true then print the metric for every epoch
            print(f'  Ep {epoch:3d}/{cfg["epochs"]} | '
                  f'tr {tr_loss:.4f} | val {vl:.4f} | '
                  f'acc {va:.4f} | F1 {vf1:.4f}')
        es(vl, model) # early stopping but latest version is no early stooping by setting patience =999 so never trigger the early stop
        if trial is not None:
            trial.report(vl, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned() # use to stop the trial if this trial is worse than the median of other finsiht trial reported value
        if es.stop:
            if verbose: print(f'  Early stop at epoch {epoch}')
            break

    return es.load_best(model), history

# used for final training only
def train_nn_sharded(cfg, shard_paths, feat_indices,
                     save_path=None, verbose=True):
    """
    Final training on all merged shards (train + val combined).
    No val evaluation — val data is merged into training, so no held-out set exists.
    Trains for exactly cfg['epochs'] epochs with no early stopping.
    """
    save_path  = save_path or os.path.join(OUTPUT_DIR, 'tmp_best_sharded.pt')
    rng        = np.random.default_rng(SEED)
    n_feats    = len(feat_indices) # column index position for those feature
    bs         = cfg['batch_size']

    model = MortgageNet(
        input_dim=n_feats, depth=cfg['depth'], hidden_dim=cfg['hidden_dim'],
        dropout_rate=cfg['dropout'], norm_type=cfg['norm_type'],
    ).to(DEVICE)
    opt  = make_optimizer(model, cfg['lr'], cfg['weight_decay']) # call the optimizer adamW ,this function is to seperate what need to apply weight decay what shouldnt
    crit = nn.CrossEntropyLoss() # stanndard cross entropy loss function
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'], eta_min=1e-5) # use consine schdular ,the lowest lr=0.00001
    history    = {'tr_loss': []}
    prefetcher = ChunkPrefetcher(feat_indices) # call the prefetcher

    print('  Caching shard sizes ...')
    shard_sizes = [] # use to store how many train row in each train shard merged file
    for p in shard_paths: # the 10 train shard merged
        with h5py.File(p, 'r') as f:
            shard_sizes.append(f['y'].shape[0])
    for p, s in zip(shard_paths, shard_sizes):
        print(f'    {os.path.basename(p):42s} {s:>12,} rows')
    print(f'  Total training rows: {sum(shard_sizes):,}')

    for epoch in range(1, cfg['epochs'] + 1): # for final traning fixed epoch = 5
        order          = rng.permutation(len(shard_paths)) # random shuffle for each shard file to random shuffle for each file
        total, n_total = 0.0, 0

        # Tracking variables for the epoch
        t_wait_total   = 0.0
        t_train_total  = 0.0
        ep_t0          = time.time()

        chunks = []
        for s_idx in order:
            p = shard_paths[s_idx] # file path for this shard
            n = shard_sizes[s_idx] # how manyu rows this shard has
            for start in range(0, n, CHUNK_ROWS):
                chunks.append((p, start, min(start + CHUNK_ROWS, n))) # [(shard1_path,0,60000000),...(shardn_path,120000000,180000000)]

        prefetcher.start(*chunks[0]) # start to fetch the first chunk

        for ci, (path, start, end) in enumerate(chunks):
            # Time the wait for the prefetcher
            t0 = time.time()
            X_np, y_np = prefetcher.get()
            t_get = time.time() - t0
            t_wait_total += t_get

            if ci + 1 < len(chunks):
                prefetcher.start(*chunks[ci + 1])

            X_tr = torch.from_numpy(X_np).to(DEVICE)
            y_tr = torch.from_numpy(y_np).to(DEVICE)
            del X_np, y_np; gc.collect()

            n_chunk = X_tr.shape[0]
            perm    = torch.randperm(n_chunk, device=DEVICE) # within file random file

            # Time the GPU training loop
            t0 = time.time()
            model.train()
            for i in range(0, n_chunk - bs + 1, bs):
                idx  = perm[i:i + bs]
                X_b  = X_tr[idx]; y_b = y_tr[idx]
                opt.zero_grad()
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                                    enabled=DEVICE.type == 'cuda'):
                    logits = model(X_b)
                    loss   = crit(logits, y_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step()
                total   += loss.item() * bs
                n_total += bs
            t_train = time.time() - t0
            t_train_total += t_train

            # Time the Garbage Collection
            t0 = time.time()
            del X_tr, y_tr, perm; gc.collect()
            t_gc = time.time() - t0

            # Time the VRAM clear
            t0 = time.time()
            torch.cuda.empty_cache()
            t_cache = time.time() - t0

            # Print detailed timings per chunk
            print(f'    [chunk {ci + 1:>2d}/{len(chunks)}] '
                  f'get={t_get:.2f}s  train={t_train:.1f}s  '
                  f'gc={t_gc:.2f}s  cache={t_cache:.2f}s  '
                  f'({os.path.basename(path)})', flush=True)

        tr_loss = total / n_total
        history['tr_loss'].append(tr_loss)
        scheduler.step()


        # Print summary for the epoch
        ep_total = time.time() - ep_t0
        if verbose:
            print(f'  === Epoch {epoch:3d}/{cfg["epochs"]} | '
                  f'tr_loss {tr_loss:.4f} | ep {ep_total:.0f}s '
                  f'[wait={t_wait_total:.0f}s train={t_train_total:.0f}s] ===', flush=True)

    torch.save(model.state_dict(), save_path)
    return model, history

In [ ]:
def _zero_layer_cfg(lr, wd, epochs=10, patience=999):
    return dict(depth=0, hidden_dim=1, dropout=0.0, norm_type='none',
                lr=lr, weight_decay=wd,
                batch_size=16384, epochs=epochs, patience=patience)


def run_zero_layer_hpo(X_tr, y_tr, X_val, y_val, wd_grid):
    """Grid search over wd_grid with lr fixed. All data are in-VRAM tensors."""
    print(f'  0-layer HPO: {len(wd_grid)} wd trials  (lr fixed at {FIXED_LR:.0e})')
    rows = []

    for wd in wd_grid:
        lr  = FIXED_LR
        cfg = _zero_layer_cfg(lr, wd, epochs=10, patience=999)
        print(f'    lr={lr:.0e}  wd={wd:.0e} ...', end=' ', flush=True)

        # Calls the in-VRAM train_nn function
        _, hist = train_nn(cfg, X_tr, y_tr, X_val, y_val,
                           save_path=os.path.join(
                               OUTPUT_DIR, f'tmp_zl_lr{lr}_wd{wd}.pt'),
                           verbose=False)

        best       = min(hist['val_loss'])
        best_ep    = hist['val_loss'].index(best) + 1
        epochs_run = len(hist['val_loss'])
        last_val   = hist['val_loss'][-1]  # ← ADD THIS

        print(f'val_loss={best:.5f}  last_val={last_val:.5f}  best_epoch={best_ep}  '
              f'stopped_at={epochs_run}/{cfg["epochs"]}')

        rows.append({'lr': lr, 'weight_decay': wd, 'val_loss': best,'last_val': last_val,
                     'best_epoch': best_ep, 'epochs_run': epochs_run})

    gc.collect()
    df       = pd.DataFrame(rows).sort_values('val_loss')
    best_row = df.iloc[0]

    print(f'\n  HPO grid results:\n{df.to_string(index=False)}')
    print(f'\n  Best: lr={best_row["lr"]:.0e}  wd={best_row["weight_decay"]:.0e}  '
          f'val_loss={best_row["val_loss"]:.5f}')

    return float(best_row['lr']), float(best_row['weight_decay']), df


def evaluate_and_save(model_name, y_true, y_pred, y_proba, current_states):
    tag = model_name.lower().replace(' ', '_')
    m   = compute_metrics(y_true, y_pred, y_proba, name=model_name)
    pc  = compute_per_class_metrics(y_true, y_pred, y_proba)
    pc.to_csv(os.path.join(OUTPUT_DIR, f'per_class_{tag}.csv'))
    print(f'\n  Per-class metrics ({model_name}):\n{pc.round(4).to_string()}')
    T, M_auc = None, None
    if current_states is not None:
        T = build_transition_matrix(current_states, y_pred)
        plot_transition_matrix(
            T, f'{model_name} — Predicted P(s\'|s)  Test 2021–2025',
            os.path.join(OUTPUT_DIR, f'transition_matrix_{tag}.png'))
        pd.DataFrame(T, index=STATE_NAMES, columns=STATE_NAMES).to_csv(
            os.path.join(OUTPUT_DIR, f'transition_matrix_{tag}.csv'))
        M_auc = compute_auc_matrix(y_true, y_proba, current_states)
        plot_auc_matrix(
            M_auc, f'7×7 AUC Matrix — {model_name}  Test 2021–2025',
            os.path.join(OUTPUT_DIR, f'auc_matrix_{tag}.png'))
        pd.DataFrame(M_auc, index=STATE_NAMES, columns=STATE_NAMES).to_csv(
            os.path.join(OUTPUT_DIR, f'auc_matrix_{tag}.csv'))
        print(f'\n  AUC matrix ({model_name}):')
        print(pd.DataFrame(M_auc, index=STATE_NAMES,
                            columns=STATE_NAMES).round(3).to_string())
    return m, T, M_auc

In [ ]:
# # ── Checkpoint helpers ────────────────────────────────────────────────────────
# CKPT_FILE = os.path.join(OUTPUT_DIR, 'script3_checkpoint.json')

# def ckpt_load():
#     if os.path.exists(CKPT_FILE):
#         with open(CKPT_FILE) as f:
#             c = json.load(f)
#         print(f'[CKPT] Resuming: {c}')
#         return c
#     return {}

# def ckpt_save(state):
#     with open(CKPT_FILE, 'w') as f:
#         json.dump(state, f, indent=2)
#     print(f'[CKPT] Saved checkpoint: {list(state.keys())}', flush=True)


# if __name__ == '__main__':
#     print('=' * 70)
#     print('Script 3 --- Train & Evaluate   (full dataset)')
#     print('Train 1999-2016 | Val 2017-2020 | Test 2021-2025')
#     print(f'SCRATCH : {SCRATCH}')
#     print(f'OUTPUT  : {OUTPUT_DIR}')
#     print('=' * 70)

#     # ---- Feature columns ----
#     with open(COLS_JSON) as f:
#         FEATURE_COLS = json.load(f)
#     MODEL_FEAT_IDX = get_model_feature_indices(FEATURE_COLS)
#     N_FEATURES     = len(MODEL_FEAT_IDX)
#     print(f'\n  HDF5 columns   : {len(FEATURE_COLS)}')
#     print(f'  Model features : {N_FEATURES}  (dropped: {REPORTING_PERIOD_COL})')

#     # ---- Recover current state for test (state_* columns only) ----
#     print('\nRecovering current state for test set (state columns only) ...')
#     curr_test = recover_current_state_from_cols(TEST_H5, FEATURE_COLS)

#     all_results, all_T_matrices, all_AUC_matrices = {}, {}, {}
#     crit_eval = nn.CrossEntropyLoss()

#     # ---- Load checkpoint ------------------------------------------------
#     ckpt = ckpt_load()

#     # ==========================================================================
#     # MODEL 1 --- CONSTANT TRANSITION BASELINE
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('MODEL 1 --- CONSTANT TRANSITION BASELINE')
#     print('=' * 70)

#     if not ckpt.get('baseline_done'):
#         baseline = ConstantBaseline()
#         try:
#             baseline.fit(MERGED_PATHS, FEATURE_COLS)
#             joblib.dump(baseline, os.path.join(OUTPUT_DIR, 'baseline.pkl'))

#             y_true_b, y_pred_b, y_proba_b, curr_b = \
#                 baseline.evaluate_on_h5(TEST_H5, FEATURE_COLS)

#             if y_true_b is not None:
#                 m_b, T_b, AUC_b = evaluate_and_save(
#                     'Baseline', y_true_b, y_pred_b, y_proba_b, curr_b)
#                 all_results['Baseline'] = m_b
#                 if T_b   is not None: all_T_matrices['Baseline']   = T_b
#                 if AUC_b is not None: all_AUC_matrices['Baseline'] = AUC_b

#                 plot_transition_matrix(
#                     baseline.T,
#                     "Baseline --- Empirical P(s'|s) from Training Data (1999-2020)",
#                     os.path.join(OUTPUT_DIR, 'transition_matrix_baseline_train.png'))
#                 pd.DataFrame(baseline.T, index=STATE_NAMES,
#                              columns=STATE_NAMES).to_csv(
#                     os.path.join(OUTPUT_DIR, 'transition_matrix_baseline_train.csv'))
#             else:
#                 print('[WARN] Baseline: current-state recovery failed.')
#         except RuntimeError as e:
#             print(f'[WARN] Baseline failed: {e}')
#         gc.collect()
#         ckpt['baseline_done'] = True; ckpt_save(ckpt)

#     else:
#         print('[CKPT] Baseline already done — reloading and re-evaluating ...')
#         baseline = joblib.load(os.path.join(OUTPUT_DIR, 'baseline.pkl'))
#         y_true_b, y_pred_b, y_proba_b, curr_b = \
#             baseline.evaluate_on_h5(TEST_H5, FEATURE_COLS)
#         if y_true_b is not None:
#             m_b, T_b, AUC_b = evaluate_and_save(
#                 'Baseline', y_true_b, y_pred_b, y_proba_b, curr_b)
#             all_results['Baseline'] = m_b
#             if T_b   is not None: all_T_matrices['Baseline']   = T_b
#             if AUC_b is not None: all_AUC_matrices['Baseline'] = AUC_b

#     # ==========================================================================
#     # PHASE 1 --- HPO FOR BOTH MODELS (HPO tensors loaded into VRAM once)
#     # Push once, reuse across all ZL and MortgageNet trials.
#     # VRAM: 60M x 171 x 4 + 20M x 171 x 4 = 54.8 GB  --> fits in H100 80 GB.
#     # Must free before final training (final training needs ~41 GB for chunks).
#     # Skip entirely if both HPOs already done.
#     # ==========================================================================
#     _hpo_needed = not ckpt.get('zl_hpo_done') or not ckpt.get('mn_hpo_done')

#     if _hpo_needed:
#         print('\n' + '=' * 70)
#         print('PHASE 1 --- Loading HPO data into VRAM')
#         print(f'  Train : {HPO_ROWS:,} rows from {os.path.basename(HPO_SHARD)}')
#         print(f'  Val   : {HPO_VAL_ROWS:,} rows from {os.path.basename(VAL_H5)}')
#         print('=' * 70)

#         print('\nLoading HPO train data ...')
#         X_hpo_np, y_hpo_np = _load_one(HPO_SHARD, MODEL_FEAT_IDX, max_rows=HPO_ROWS)
#         print('Loading HPO val data ...')
#         X_val_np, y_val_np = _load_one(VAL_H5, MODEL_FEAT_IDX, max_rows=HPO_VAL_ROWS)

#         # Push HPO data to GPU (stays there for all HPO trials)
#         print('Pushing HPO data to GPU ...')
#         X_tr_gpu  = torch.from_numpy(X_hpo_np).to(DEVICE)
#         y_tr_gpu  = torch.from_numpy(y_hpo_np).to(DEVICE)
#         X_val_gpu = torch.from_numpy(X_val_np).to(DEVICE)
#         y_val_gpu = torch.from_numpy(y_val_np).to(DEVICE)
#         del X_hpo_np, y_hpo_np, X_val_np, y_val_np; gc.collect()
#         if DEVICE.type == 'cuda':
#             used  = torch.cuda.memory_allocated() / 1e9
#             total = torch.cuda.get_device_properties(0).total_memory / 1e9
#             print(f'  VRAM after HPO data push: {used:.1f} / {total:.1f} GB')
#     else:
#         print('\n[CKPT] Both HPOs already done — skipping VRAM data load')

#     # ==========================================================================
#     # HPO --- MODEL 2: ZERO-HIDDEN-LAYER NN
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('HPO --- MODEL 2: ZERO-HIDDEN-LAYER NN  (grid search)')
#     print('=' * 70)

#     if not ckpt.get('zl_hpo_done'):
#         best_lr_zl, best_wd_zl, zl_hpo_df = run_zero_layer_hpo(
#             X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu, ZERO_LAYER_WD_GRID)
#         for _f in glob.glob(os.path.join(OUTPUT_DIR, 'tmp_zl_*.pt')):
#             os.remove(_f)
#         zl_hpo_df.to_csv(
#             os.path.join(OUTPUT_DIR, 'zero_layer_hpo_results.csv'), index=False)
#         ckpt['zl_hpo_done'] = True
#         ckpt['best_lr_zl']  = best_lr_zl
#         ckpt['best_wd_zl']  = best_wd_zl
#         ckpt_save(ckpt)
#     else:
#         best_lr_zl = ckpt['best_lr_zl']
#         best_wd_zl = ckpt['best_wd_zl']
#         print(f'[CKPT] ZL HPO already done — best_lr={best_lr_zl}, best_wd={best_wd_zl}')

#     # ==========================================================================
#     # HPO --- MODEL 3: MORTGAGENET (15 Optuna trials)
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('HPO --- MODEL 3: MORTGAGENET  (15 Optuna trials)')
#     print('=' * 70)

#     if not ckpt.get('mn_hpo_done'):
#         HPO_EPOCHS, HPO_PATIENCE = 10, 999

#         def objective(trial):
#             cfg = dict(
#                 depth        = trial.suggest_categorical('depth',      [3, 5, 7]),
#                 hidden_dim   = trial.suggest_categorical('hidden_dim', [128, 256, 512]),
#                 dropout      = trial.suggest_categorical('dropout',    [0.1, 0.2, 0.3]),
#                 norm_type    = trial.suggest_categorical('norm_type',  ['batch', 'layer', 'none']),
#                 lr           = trial.suggest_float('lr', 1e-4, 1e-2, log=True),
#                 weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
#                 batch_size=16384, epochs=HPO_EPOCHS, patience=HPO_PATIENCE,
#             )
#             _, hist = train_nn(
#                 cfg, X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu,
#                 save_path=os.path.join(OUTPUT_DIR, f'tmp_optuna_{trial.number}.pt'),
#                 verbose=False, trial=trial)
#             best_val = min(hist['val_loss'])
#             print(f'  Trial {trial.number:>2d} | '
#                   f'depth={cfg["depth"]} hidden={cfg["hidden_dim"]} '
#                   f'dropout={cfg["dropout"]} norm={cfg["norm_type"]} '
#                   f'wd={cfg["weight_decay"]:.2e} | '
#                   f'val_loss={best_val:.5f}', flush=True)
#             return best_val

#         if os.path.exists(OPTUNA_DB.replace('sqlite:///', '')):
#             os.remove(OPTUNA_DB.replace('sqlite:///', ''))

#         study = optuna.create_study(
#             direction      = 'minimize',
#             pruner         = optuna.pruners.MedianPruner(n_warmup_steps=5),
#             sampler        = optuna.samplers.TPESampler(seed=SEED),
#             study_name     = 'mortgage_nn_hpo_full',
#             storage        = OPTUNA_DB,
#             load_if_exists = False,
#         )
#         study.optimize(objective, n_trials=15, timeout=72000)

#         trials_df = study.trials_dataframe()
#         trials_df.to_csv(os.path.join(OUTPUT_DIR, 'optuna_trials.csv'), index=False)
#         cols = ['number', 'value', 'params_depth', 'params_hidden_dim',
#                 'params_dropout', 'params_norm_type', 'params_lr', 'params_weight_decay']
#         print('\n' + '-' * 65)
#         print('  MortgageNet HPO --- All Trials (sorted by val_loss)')
#         print('-' * 65)
#         print(trials_df[cols].sort_values('value').to_string(index=False))
#         print('-' * 65)

#         best = study.best_trial
#         print(f'\n  Best trial #{best.number}  val_loss={best.value:.5f}')
#         for k, v in study.best_params.items():
#             print(f'    {k}: {v}')
#         with open(os.path.join(OUTPUT_DIR, 'best_hpo_params.json'), 'w') as fj:
#             json.dump(study.best_params, fj, indent=2)

#         # Optuna optimisation history plot
#         completed = [t for t in study.trials if t.value is not None]
#         fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#         axes[0].plot([t.number for t in completed],
#                      [t.value  for t in completed],
#                      marker='o', markersize=4, color='steelblue')
#         axes[0].axhline(best.value, color='red', linestyle='--',
#                         label=f'Best={best.value:.5f}')
#         axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val Loss')
#         axes[0].set_title('Optuna --- Val Loss per Trial')
#         axes[0].legend(); axes[0].grid(alpha=0.3)
#         wds  = [t.params['weight_decay'] for t in completed]
#         vals = [t.value for t in completed]
#         axes[1].scatter(wds, vals, color='steelblue', alpha=0.7)
#         axes[1].set_xscale('log'); axes[1].set_xlabel('weight_decay (log scale)')
#         axes[1].set_ylabel('Val Loss')
#         axes[1].set_title('Val Loss vs Weight Decay'); axes[1].grid(alpha=0.3)
#         plt.tight_layout()
#         plt.savefig(os.path.join(OUTPUT_DIR, 'optuna_hpo_summary.png'), dpi=150)
#         plt.close()
#         print('  Saved: optuna_hpo_summary.png')

#         for _f in glob.glob(os.path.join(OUTPUT_DIR, 'tmp_optuna_*.pt')):
#             os.remove(_f)

#         mn_best_params = study.best_params
#         ckpt['mn_hpo_done']    = True
#         ckpt['mn_best_params'] = mn_best_params
#         ckpt_save(ckpt)

#     else:
#         mn_best_params = ckpt['mn_best_params']
#         print(f'[CKPT] MN HPO already done — best params: {mn_best_params}')

#     # ==========================================================================
#     # PHASE 2 --- FREE HPO GPU TENSORS BEFORE FINAL TRAINING
#     # VRAM budget during final training: chunk (~41 GB) + model + perm ~= 43 GB.
#     # HPO tensors (54.8 GB) would overflow H100 80 GB if retained.
#     # ==========================================================================
#     if _hpo_needed:
#         print('\nFreeing HPO GPU tensors ...')
#         del X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu
#         gc.collect(); torch.cuda.empty_cache()
#         if DEVICE.type == 'cuda':
#             used = torch.cuda.memory_allocated() / 1e9
#             print(f'  VRAM after free: {used:.1f} GB allocated')

#     # ==========================================================================
#     # PHASE 3a --- ZERO-HIDDEN-LAYER: FINAL TRAINING
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('FINAL --- MODEL 2: ZERO-HIDDEN-LAYER NN  (all merged shards)')
#     print('=' * 70)

#     ZL_CFG        = _zero_layer_cfg(best_lr_zl, best_wd_zl, epochs=10, patience=999)
#     ZL_MODEL_PATH = os.path.join(OUTPUT_DIR, 'zero_layer_nn.pt')
#     with open(os.path.join(OUTPUT_DIR, 'final_cfg_zero_layer.json'), 'w') as fj:
#         json.dump(ZL_CFG, fj, indent=2)

#     if not ckpt.get('zl_train_done'):
#         print('\nFinal 0-layer NN: training on all merged shards ...')
#         zl_model, zl_history = train_nn_sharded(
#             ZL_CFG, MERGED_PATHS, MODEL_FEAT_IDX,
#             save_path=ZL_MODEL_PATH, verbose=True)
#         torch.save({'model_state': zl_model.state_dict(),
#                     'cfg': ZL_CFG, 'feature_indices': MODEL_FEAT_IDX}, ZL_MODEL_PATH)
#         plot_training_curves(zl_history,
#                              os.path.join(OUTPUT_DIR, 'zero_layer_training_curves.png'))
#         ckpt['zl_train_done'] = True; ckpt_save(ckpt)
#     else:
#         print('[CKPT] ZL final training already done — reloading model ...')
#         zl_model = MortgageNet(
#             input_dim    = N_FEATURES,
#             depth        = ZL_CFG['depth'],
#             hidden_dim   = ZL_CFG['hidden_dim'],
#             dropout_rate = ZL_CFG['dropout'],
#             norm_type    = ZL_CFG['norm_type'],
#         ).to(DEVICE)
#         ckpt_data = torch.load(ZL_MODEL_PATH, map_location=DEVICE)
#         zl_model.load_state_dict(ckpt_data['model_state'])
#         zl_model.eval()

#     print('\nEvaluating 0-layer NN on test set (chunked streaming) ...')
#     _, _, _, zl_pred, zl_true, zl_proba = evaluate_h5_chunked(
#         TEST_H5, MODEL_FEAT_IDX, zl_model, crit_eval)

#     m_zl, T_zl, AUC_zl = evaluate_and_save(
#         'Logistic Regression', zl_true, zl_pred, zl_proba, curr_test)
#     all_results['Logistic Regression'] = m_zl
#     if T_zl   is not None: all_T_matrices['Logistic Regression']   = T_zl
#     if AUC_zl is not None: all_AUC_matrices['Logistic Regression'] = AUC_zl
#     del zl_model; gc.collect(); torch.cuda.empty_cache()

#     # ==========================================================================
#     # PHASE 3b --- MORTGAGENET: FINAL TRAINING
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('FINAL --- MODEL 3: MORTGAGENET  (all merged shards)')
#     print('=' * 70)

#     FINAL_CFG        = dict(**mn_best_params,
#                              epochs=10, patience=999, batch_size=16384)
#     FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_transition_nn.pt')
#     with open(os.path.join(OUTPUT_DIR, 'final_cfg_mortgagenet.json'), 'w') as fj:
#         json.dump(FINAL_CFG, fj, indent=2)

#     if not ckpt.get('mn_train_done'):
#         print('\nFinal MortgageNet: training on all merged shards ...')
#         final_model, final_history = train_nn_sharded(
#             FINAL_CFG, MERGED_PATHS, MODEL_FEAT_IDX,
#             save_path=FINAL_MODEL_PATH, verbose=True)
#         torch.save({'model_state': final_model.state_dict(),
#                     'cfg': FINAL_CFG, 'feature_indices': MODEL_FEAT_IDX},
#                    FINAL_MODEL_PATH)
#         plot_training_curves(final_history,
#                              os.path.join(OUTPUT_DIR, 'nn_training_curves.png'))
#         ckpt['mn_train_done'] = True; ckpt_save(ckpt)
#     else:
#         print('[CKPT] MN final training already done — reloading model ...')
#         final_model = MortgageNet(
#             input_dim    = N_FEATURES,
#             depth        = mn_best_params['depth'],
#             hidden_dim   = mn_best_params['hidden_dim'],
#             dropout_rate = mn_best_params['dropout'],
#             norm_type    = mn_best_params['norm_type'],
#         ).to(DEVICE)
#         ckpt_data = torch.load(FINAL_MODEL_PATH, map_location=DEVICE)
#         final_model.load_state_dict(ckpt_data['model_state'])
#         final_model.eval()

#     print('\nEvaluating MortgageNet on test set (chunked streaming) ...')
#     _, _, _, nn_pred, nn_true, nn_proba = evaluate_h5_chunked(
#         TEST_H5, MODEL_FEAT_IDX, final_model, crit_eval)

#     m_nn, T_nn, AUC_nn = evaluate_and_save(
#         'MortgageNet', nn_true, nn_pred, nn_proba, curr_test)
#     all_results['MortgageNet'] = m_nn
#     if T_nn   is not None: all_T_matrices['MortgageNet']   = T_nn
#     if AUC_nn is not None: all_AUC_matrices['MortgageNet'] = AUC_nn

#     # ==========================================================================
#     # FINAL COMPARISON TABLE
#     # ==========================================================================
#     print('\n' + '=' * 70)
#     print('FINAL COMPARISON --- TEST SET (2021-2025)')
#     print('Primary: log_loss (down)  |  Discrimination: 7x7 AUC matrix (above)')
#     print('=' * 70)

#     order      = ['Baseline', 'Logistic Regression', 'MortgageNet']
#     results_df = pd.DataFrame(
#         [{'Model': k, **all_results[k]} for k in order if k in all_results])
#     print('\n' + results_df[['Model', 'log_loss', 'macro_f1',
#                               'accuracy']].round(5).to_string(index=False))
#     results_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)

#     if len(results_df) == 3:
#         ll = dict(zip(results_df['Model'], results_df['log_loss']))
#         ok = ll['MortgageNet'] < ll['Logistic Regression'] < ll['Baseline']
#         print(f'\nPerformance ordering (log_loss down): '
#               f'{"CONFIRMED" if ok else "NOT MET"}')

#     plot_comparison_bar(results_df,
#                         os.path.join(OUTPUT_DIR, 'model_comparison_bar.png'))
#     if len(all_T_matrices) >= 2:
#         plot_transition_matrices_side_by_side(
#             all_T_matrices,
#             os.path.join(OUTPUT_DIR, 'transition_matrices_all.png'))
#     if len(all_AUC_matrices) >= 2:
#         plot_auc_matrices_side_by_side(
#             all_AUC_matrices,
#             os.path.join(OUTPUT_DIR, 'auc_matrices_all.png'))

#     print(f'\nAll artefacts saved to: {OUTPUT_DIR}')
#     print('script3_full.py  complete.')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
CKPT_FILE = os.path.join(OUTPUT_DIR, 'script3_checkpoint.json')

def ckpt_load():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE) as f:
            c = json.load(f)
        print(f'[CKPT] Resuming: {c}')
        return c
    return {}

def ckpt_save(state):
    with open(CKPT_FILE, 'w') as f:
        json.dump(state, f, indent=2)
    print(f'[CKPT] Saved checkpoint: {list(state.keys())}', flush=True)

try:
    if __name__ == '__main__':
        print('=' * 70)
        print('Script 3 --- Train & Evaluate   (full dataset)')
        print('Train 1999-2016 | Val 2017-2020 | Test 2021-2025')
        print(f'SCRATCH : {SCRATCH}')
        print(f'OUTPUT  : {OUTPUT_DIR}')
        print('=' * 70)

        # ---- Feature columns ----
        with open(COLS_JSON) as f:
            FEATURE_COLS = json.load(f)
        MODEL_FEAT_IDX = get_model_feature_indices(FEATURE_COLS)
        N_FEATURES     = len(MODEL_FEAT_IDX)
        print(f'\n  HDF5 columns   : {len(FEATURE_COLS)}')
        print(f'  Model features : {N_FEATURES}  (dropped: {REPORTING_PERIOD_COL})')

        all_results, all_T_matrices, all_AUC_matrices = {}, {}, {}
        crit_eval = nn.CrossEntropyLoss()

        # ---- Load checkpoint ------------------------------------------------
        ckpt = ckpt_load()

        # ==========================================================================
        # MODEL 1 --- CONSTANT TRANSITION BASELINE
        # ==========================================================================
        print('\n' + '=' * 70)
        print('MODEL 1 --- CONSTANT TRANSITION BASELINE')
        print('=' * 70)

        if not ckpt.get('baseline_done'):
            baseline = ConstantBaseline()
            try:
                baseline.fit(MERGED_PATHS, FEATURE_COLS)
                joblib.dump(baseline, os.path.join(OUTPUT_DIR, 'baseline.pkl'))
                gc.collect()
                ctypes.CDLL("libc.so.6").malloc_trim(0)

                #print('\nRecovering current state for test set ...')
                #curr_test = recover_current_state_from_cols(TEST_H5, FEATURE_COLS)
                #gc.collect()

                y_true_b, y_pred_b, y_proba_b, curr_b = \
                    baseline.evaluate_on_h5(TEST_H5, FEATURE_COLS)
                curr_test = curr_b

                if y_true_b is not None:
                    m_b, T_b, AUC_b = evaluate_and_save(
                        'Baseline', y_true_b, y_pred_b, y_proba_b, curr_b)
                    all_results['Baseline'] = m_b
                    if T_b   is not None: all_T_matrices['Baseline']   = T_b
                    if AUC_b is not None: all_AUC_matrices['Baseline'] = AUC_b

                    plot_transition_matrix(
                        baseline.T,
                        "Baseline --- Empirical P(s'|s) from Training Data (1999-2020)",
                        os.path.join(OUTPUT_DIR, 'transition_matrix_baseline_train.png'))
                    pd.DataFrame(baseline.T, index=STATE_NAMES,
                                columns=STATE_NAMES).to_csv(
                        os.path.join(OUTPUT_DIR, 'transition_matrix_baseline_train.csv'))
                    try:
                        del y_true_b, y_pred_b, y_proba_b
                    except NameError:
                        pass
                    gc.collect()
                    ctypes.CDLL("libc.so.6").malloc_trim(0)
                else:
                    print('[WARN] Baseline: current-state recovery failed.')
            except RuntimeError as e:
                print(f'[WARN] Baseline failed: {e}')
            gc.collect()
            ckpt['baseline_done'] = True; ckpt_save(ckpt)

        else:
            print('[CKPT] Baseline already done — reloading and re-evaluating ...')
            baseline = joblib.load(os.path.join(OUTPUT_DIR, 'baseline.pkl'))
            gc.collect()
            ctypes.CDLL("libc.so.6").malloc_trim(0)

            y_true_b, y_pred_b, y_proba_b, curr_b = \
                baseline.evaluate_on_h5(TEST_H5, FEATURE_COLS)
            curr_test = curr_b
            if y_true_b is not None:
                m_b, T_b, AUC_b = evaluate_and_save(
                    'Baseline', y_true_b, y_pred_b, y_proba_b, curr_b)
                all_results['Baseline'] = m_b
                if T_b   is not None: all_T_matrices['Baseline']   = T_b
                if AUC_b is not None: all_AUC_matrices['Baseline'] = AUC_b

                try:
                    del y_true_b, y_pred_b, y_proba_b
                except NameError:
                    pass
                gc.collect()
                ctypes.CDLL("libc.so.6").malloc_trim(0)
                print('Baseline test arrays freed.')


        # ==========================================================================
        # PHASE 1 --- HPO FOR BOTH MODELS (HPO tensors loaded into VRAM once)
        # Push once, reuse across all ZL and MortgageNet trials.
        # VRAM: 60M x 171 x 4 + 20M x 171 x 4 = 54.8 GB  --> fits in H100 80 GB.
        # Must free before final training (final training needs ~41 GB for chunks).
        # Skip entirely if both HPOs already done.
        # ==========================================================================
        _hpo_needed = not ckpt.get('zl_hpo_done') or not ckpt.get('mn_hpo_done')

        if _hpo_needed:
            print('\n' + '=' * 70)
            print('PHASE 1 --- Loading HPO data into VRAM')
            print(f'  Train : {HPO_ROWS:,} rows from {os.path.basename(HPO_SHARD)}')
            print(f'  Val   : {HPO_VAL_ROWS:,} rows from {os.path.basename(VAL_H5)}')
            print('=' * 70)

            print('\nLoading HPO train data ...')
            X_hpo_np, y_hpo_np = _load_one(HPO_SHARD, MODEL_FEAT_IDX, max_rows=HPO_ROWS)
            print('Loading HPO val data ...')
            X_val_np, y_val_np = _load_one(VAL_H5, MODEL_FEAT_IDX, max_rows=HPO_VAL_ROWS)

            # Push HPO data to GPU (stays there for all HPO trials)
            print('Pushing HPO data to GPU ...')
            X_tr_gpu  = torch.from_numpy(X_hpo_np).to(DEVICE)
            y_tr_gpu  = torch.from_numpy(y_hpo_np).to(DEVICE)
            X_val_gpu = torch.from_numpy(X_val_np).to(DEVICE)
            y_val_gpu = torch.from_numpy(y_val_np).to(DEVICE)
            del X_hpo_np, y_hpo_np, X_val_np, y_val_np; gc.collect()
            if DEVICE.type == 'cuda':
                used  = torch.cuda.memory_allocated() / 1e9
                total = torch.cuda.get_device_properties(0).total_memory / 1e9
                print(f'  VRAM after HPO data push: {used:.1f} / {total:.1f} GB')
        else:
            print('\n[CKPT] Both HPOs already done — skipping VRAM data load')

        # ==========================================================================
        # HPO --- MODEL 2: ZERO-HIDDEN-LAYER NN
        # ==========================================================================
        print('\n' + '=' * 70)
        print('HPO --- MODEL 2: ZERO-HIDDEN-LAYER NN  (grid search)')
        print('=' * 70)

        if not ckpt.get('zl_hpo_done'):
            best_lr_zl, best_wd_zl, zl_hpo_df = run_zero_layer_hpo(
                X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu, ZERO_LAYER_WD_GRID)
            for _f in glob.glob(os.path.join(OUTPUT_DIR, 'tmp_zl_*.pt')):
                os.remove(_f)
            zl_hpo_df.to_csv(
                os.path.join(OUTPUT_DIR, 'zero_layer_hpo_results.csv'), index=False)
            ckpt['zl_hpo_done'] = True
            ckpt['best_lr_zl']  = best_lr_zl
            ckpt['best_wd_zl']  = best_wd_zl
            ckpt_save(ckpt)
        else:
            best_lr_zl = ckpt['best_lr_zl']
            best_wd_zl = ckpt['best_wd_zl']
            print(f'[CKPT] ZL HPO already done — best_lr={best_lr_zl}, best_wd={best_wd_zl}')

        # ==========================================================================
        # HPO --- MODEL 3: MORTGAGENET (15 Optuna trials)
        # ==========================================================================
        print('\n' + '=' * 70)
        print('HPO --- MODEL 3: MORTGAGENET  (15 Optuna trials)')
        print('=' * 70)

        if not ckpt.get('mn_hpo_done'):
            HPO_EPOCHS, HPO_PATIENCE = 10, 999

            def objective(trial):
                cfg = dict(
                    depth        = trial.suggest_categorical('depth',      [3, 5, 7]),
                    hidden_dim   = trial.suggest_categorical('hidden_dim', [128, 256, 512]),
                    dropout      = trial.suggest_categorical('dropout',    [0.1, 0.2, 0.3]),
                    norm_type    = trial.suggest_categorical('norm_type',  ['batch', 'layer', 'none']),
                    lr = 1e-4,
                    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
                    batch_size=16384, epochs=HPO_EPOCHS, patience=HPO_PATIENCE,
                )
                _, hist = train_nn(
                    cfg, X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu,
                    save_path=os.path.join(OUTPUT_DIR, f'tmp_optuna_{trial.number}.pt'),
                    verbose=False, trial=trial)
                best_val = min(hist['val_loss'])
                last_val = hist['val_loss'][-1]
                print(f'  Trial {trial.number:>2d} | '
                      f'depth={cfg["depth"]} hidden={cfg["hidden_dim"]} '
                      f'dropout={cfg["dropout"]} norm={cfg["norm_type"]} '
                      f'wd={cfg["weight_decay"]:.2e} | '
                      f'best_val={best_val:.5f} last_val={last_val:.5f}', flush=True)
                return best_val

            if os.path.exists(OPTUNA_DB.replace('sqlite:///', '')):
                os.remove(OPTUNA_DB.replace('sqlite:///', ''))

            study = optuna.create_study(
                direction      = 'minimize',
                pruner         = optuna.pruners.MedianPruner(n_warmup_steps=5),
                sampler        = optuna.samplers.TPESampler(seed=SEED),
                study_name     = 'mortgage_nn_hpo_full',
                storage        = OPTUNA_DB,
                load_if_exists = False,
            )
            study.optimize(objective, n_trials=15, timeout=72000)

            trials_df = study.trials_dataframe()
            trials_df.to_csv(os.path.join(OUTPUT_DIR, 'optuna_trials.csv'), index=False)
            cols = ['number', 'value', 'params_depth', 'params_hidden_dim',
                    'params_dropout', 'params_norm_type', 'params_weight_decay']
            print('\n' + '-' * 65)
            print('  MortgageNet HPO --- All Trials (sorted by val_loss)')
            print('-' * 65)
            print(trials_df[cols].sort_values('value').to_string(index=False))
            print('-' * 65)

            best = study.best_trial
            print(f'\n  Best trial #{best.number}  val_loss={best.value:.5f}')
            for k, v in study.best_params.items():
                print(f'    {k}: {v}')
            with open(os.path.join(OUTPUT_DIR, 'best_hpo_params.json'), 'w') as fj:
                json.dump(study.best_params, fj, indent=2)

            # Optuna optimisation history plot
            completed = [t for t in study.trials if t.value is not None]
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot([t.number for t in completed],
                        [t.value  for t in completed],
                        marker='o', markersize=4, color='steelblue')
            axes[0].axhline(best.value, color='red', linestyle='--',
                            label=f'Best={best.value:.5f}')
            axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val Loss')
            axes[0].set_title('Optuna --- Val Loss per Trial')
            axes[0].legend(); axes[0].grid(alpha=0.3)
            wds  = [t.params['weight_decay'] for t in completed]
            vals = [t.value for t in completed]
            axes[1].scatter(wds, vals, color='steelblue', alpha=0.7)
            axes[1].set_xscale('log'); axes[1].set_xlabel('weight_decay (log scale)')
            axes[1].set_ylabel('Val Loss')
            axes[1].set_title('Val Loss vs Weight Decay'); axes[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, 'optuna_hpo_summary.png'), dpi=150)
            plt.close()
            print('  Saved: optuna_hpo_summary.png')

            for _f in glob.glob(os.path.join(OUTPUT_DIR, 'tmp_optuna_*.pt')):
                os.remove(_f)

            mn_best_params = study.best_params
            ckpt['mn_hpo_done']    = True
            ckpt['mn_best_params'] = mn_best_params
            ckpt_save(ckpt)

        else:
            mn_best_params = ckpt['mn_best_params']
            print(f'[CKPT] MN HPO already done — best params: {mn_best_params}')

        # ==========================================================================
        # PHASE 2 --- FREE HPO GPU TENSORS BEFORE FINAL TRAINING
        # VRAM budget during final training: chunk (~41 GB) + model + perm ~= 43 GB.
        # HPO tensors (54.8 GB) would overflow H100 80 GB if retained.
        # ==========================================================================
        if _hpo_needed:
            print('\nFreeing HPO GPU tensors ...')
            del X_tr_gpu, y_tr_gpu, X_val_gpu, y_val_gpu
            gc.collect(); torch.cuda.empty_cache()
            if DEVICE.type == 'cuda':
                used = torch.cuda.memory_allocated() / 1e9
                print(f'  VRAM after free: {used:.1f} GB allocated')

        # ==========================================================================
        # PHASE 3a --- ZERO-HIDDEN-LAYER: FINAL TRAINING
        # ==========================================================================
        print('\n' + '=' * 70)
        print('FINAL --- MODEL 2: ZERO-HIDDEN-LAYER NN  (all merged shards)')
        print('=' * 70)

        ZL_CFG        = _zero_layer_cfg(best_lr_zl, best_wd_zl, epochs=5, patience=999)
        ZL_MODEL_PATH = os.path.join(OUTPUT_DIR, 'zero_layer_nn.pt')
        with open(os.path.join(OUTPUT_DIR, 'final_cfg_zero_layer.json'), 'w') as fj:
            json.dump(ZL_CFG, fj, indent=2)

        if not ckpt.get('zl_train_done'):
            print('\nFinal 0-layer NN: training on all merged shards ...')
            zl_model, zl_history = train_nn_sharded(
                ZL_CFG, MERGED_PATHS, MODEL_FEAT_IDX,
                save_path=ZL_MODEL_PATH, verbose=True)
            torch.save({'model_state': zl_model.state_dict(),
                        'cfg': ZL_CFG, 'feature_indices': MODEL_FEAT_IDX}, ZL_MODEL_PATH)
            plot_training_curves(zl_history,
                                os.path.join(OUTPUT_DIR, 'zero_layer_training_curves.png'))
            ckpt['zl_train_done'] = True; ckpt_save(ckpt)
            gc.collect()
            torch.cuda.synchronize()           # ← ADD
            torch.cuda.empty_cache()
        else:
            print('[CKPT] ZL final training already done — reloading model ...')
            zl_model = MortgageNet(
                input_dim    = N_FEATURES,
                depth        = ZL_CFG['depth'],
                hidden_dim   = ZL_CFG['hidden_dim'],
                dropout_rate = ZL_CFG['dropout'],
                norm_type    = ZL_CFG['norm_type'],
            ).to(DEVICE)
            ckpt_data = torch.load(ZL_MODEL_PATH, map_location=DEVICE)
            zl_model.load_state_dict(ckpt_data['model_state'])
            zl_model.eval()

            gc.collect()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

        print('\nEvaluating 0-layer NN on test set (chunked streaming) ...')
        _, _, _, zl_pred, zl_true, zl_proba = evaluate_h5_chunked(
            TEST_H5, MODEL_FEAT_IDX, zl_model, crit_eval)

        m_zl, T_zl, AUC_zl = evaluate_and_save(
            'Logistic Regression', zl_true, zl_pred, zl_proba, curr_test)
        all_results['Logistic Regression'] = m_zl
        if T_zl   is not None: all_T_matrices['Logistic Regression']   = T_zl
        if AUC_zl is not None: all_AUC_matrices['Logistic Regression'] = AUC_zl
        del zl_model, zl_pred, zl_true, zl_proba
        gc.collect(); torch.cuda.empty_cache()
        ctypes.CDLL("libc.so.6").malloc_trim(0)

        # ==========================================================================
        # PHASE 3b --- MORTGAGENET: FINAL TRAINING
        # ==========================================================================
        print('\n' + '=' * 70)
        print('FINAL --- MODEL 3: MORTGAGENET  (all merged shards)')
        print('=' * 70)

        FINAL_CFG        = dict(**mn_best_params,lr=1e-4,
                                epochs=5, patience=999, batch_size=16384)
        FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_transition_nn.pt')
        with open(os.path.join(OUTPUT_DIR, 'final_cfg_mortgagenet.json'), 'w') as fj:
            json.dump(FINAL_CFG, fj, indent=2)

        if not ckpt.get('mn_train_done'):
            print('\nFinal MortgageNet: training on all merged shards ...')
            final_model, final_history = train_nn_sharded(
                FINAL_CFG, MERGED_PATHS, MODEL_FEAT_IDX,
                save_path=FINAL_MODEL_PATH, verbose=True)
            torch.save({'model_state': final_model.state_dict(),
                        'cfg': FINAL_CFG, 'feature_indices': MODEL_FEAT_IDX},
                      FINAL_MODEL_PATH)
            plot_training_curves(final_history,
                                os.path.join(OUTPUT_DIR, 'nn_training_curves.png'))
            ckpt['mn_train_done'] = True; ckpt_save(ckpt)
            gc.collect()
            torch.cuda.synchronize()           # ← ADD
            torch.cuda.empty_cache()
        else:
            print('[CKPT] MN final training already done — reloading model ...')
            final_model = MortgageNet(
                input_dim    = N_FEATURES,
                depth        = mn_best_params['depth'],
                hidden_dim   = mn_best_params['hidden_dim'],
                dropout_rate = mn_best_params['dropout'],
                norm_type    = mn_best_params['norm_type'],
            ).to(DEVICE)
            ckpt_data = torch.load(FINAL_MODEL_PATH, map_location=DEVICE)
            final_model.load_state_dict(ckpt_data['model_state'])
            final_model.eval()
            gc.collect()
            torch.cuda.synchronize()            # ← ADD
            torch.cuda.empty_cache()

        print('\nEvaluating MortgageNet on test set (chunked streaming) ...')
        _, _, _, nn_pred, nn_true, nn_proba = evaluate_h5_chunked(
            TEST_H5, MODEL_FEAT_IDX, final_model, crit_eval)

        m_nn, T_nn, AUC_nn = evaluate_and_save(
            'MortgageNet', nn_true, nn_pred, nn_proba, curr_test)
        all_results['MortgageNet'] = m_nn
        if T_nn   is not None: all_T_matrices['MortgageNet']   = T_nn
        if AUC_nn is not None: all_AUC_matrices['MortgageNet'] = AUC_nn
        del final_model, nn_pred, nn_true, nn_proba, curr_test
        gc.collect(); torch.cuda.empty_cache()
        ctypes.CDLL("libc.so.6").malloc_trim(0)

        # ==========================================================================
        # FINAL COMPARISON TABLE
        # ==========================================================================
        print('\n' + '=' * 70)
        print('FINAL COMPARISON --- TEST SET (2021-2025)')
        print('Primary: log_loss (down)  |  Discrimination: 7x7 AUC matrix (above)')
        print('=' * 70)

        order      = ['Baseline', 'Logistic Regression', 'MortgageNet']
        results_df = pd.DataFrame(
            [{'Model': k, **all_results[k]} for k in order if k in all_results])
        print('\n' + results_df[['Model', 'log_loss', 'macro_f1',
                                  'accuracy']].round(5).to_string(index=False))
        results_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)

        if len(results_df) == 3:
            ll = dict(zip(results_df['Model'], results_df['log_loss']))
            ok = ll['MortgageNet'] < ll['Logistic Regression'] < ll['Baseline']
            print(f'\nPerformance ordering (log_loss down): '
                  f'{"CONFIRMED" if ok else "NOT MET"}')

        plot_comparison_bar(results_df,
                            os.path.join(OUTPUT_DIR, 'model_comparison_bar.png'))
        if len(all_T_matrices) >= 2:
            plot_transition_matrices_side_by_side(
                all_T_matrices,
                os.path.join(OUTPUT_DIR, 'transition_matrices_all.png'))
        if len(all_AUC_matrices) >= 2:
            plot_auc_matrices_side_by_side(
                all_AUC_matrices,
                os.path.join(OUTPUT_DIR, 'auc_matrices_all.png'))

        print(f'\nAll artefacts saved to: {OUTPUT_DIR}')
        print('script3_full.py  complete.')


except Exception as e:
    import traceback
    error_msg = traceback.format_exc()
    with open(os.path.join(OUTPUT_DIR, 'CRASH_LOG.txt'), 'w') as f:
        f.write(error_msg)
    print(f'CRASHED: {error_msg}', flush=True)

finally:
    print('Releasing runtime to stop compute billing...', flush=True)
    from google.colab import runtime
    runtime.unassign()




Script 3 --- Train & Evaluate   (full dataset)
Train 1999-2016 | Val 2017-2020 | Test 2021-2025
SCRATCH : /mnt/local-scratch
OUTPUT  : /content/drive/MyDrive/dissertation/models/full_run

  HDF5 columns   : 175
  Model features : 174  (dropped: Reporting_Period_Int)
[CKPT] Resuming: {'baseline_done': True, 'zl_hpo_done': True, 'best_lr_zl': 0.001, 'best_wd_zl': 1e-05, 'zl_train_done': True}

MODEL 1 --- CONSTANT TRANSITION BASELINE
[CKPT] Baseline already done — reloading and re-evaluating ...

Baseline — Scalar Metrics:
  Neg. avg log-likelihood (↓) : 0.09533 
  Macro-F1                    : 0.42131 
  Accuracy                    : 0.97967 

  Per-class metrics (Baseline):
             precision  recall      f1    support
Current         0.9869  0.9954  0.9911  701547947
D30             0.3828  0.3800  0.3814    5669045
D60             0.0000  0.0000  0.0000    1367278
D90+            0.5189  0.9788  0.6782    1572431
Foreclosure     0.8796  0.9182  0.8985    3429649
REO             0